# Job Posting Summarizer

Day 1 community contribution, a commercial summarization tool for job seekers.

Paste a job posting (or scrape a URL when the site allows it) and get a structured brief:
role, requirements, skills, fit profile, red flags, and an apply checklist.

Works across Indonesian and international platforms (JobStreet, Glints, Kalibrr, LinkedIn, Indeed, Greenhouse, Lever, etc.).

**Requires:** `.env` with `OPENAI_API_KEY` (see course setup).

In [ ]:
import os
import sys
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

WEEK1_DIR = Path("../..").resolve()
if str(WEEK1_DIR) not in sys.path:
    sys.path.insert(0, str(WEEK1_DIR))

load_dotenv(override=True)
openai = OpenAI(base_url="https://ai.sumopod.com/v1")
MODEL = "gpt-4.1-mini"

In [7]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found — add OPENAI_API_KEY to your .env file.")
elif api_key.strip() != api_key:
    print("API key has extra whitespace — trim it in .env.")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


## Prompts

Same pattern as Day 1: a **system prompt** defines role and output format; a **user prompt** carries the job posting text.

In [8]:
system_prompt = """
You are a career assistant that analyzes job postings scraped from job boards
(JobStreet, Glints, Kalibrr, LinkedIn, Indeed, Greenhouse, Lever, etc.).

Your job is to extract structured, actionable information for a job seeker.
The input may be messy (navigation text, ads, duplicate content, mixed Indonesian/English).
Ignore irrelevant website chrome and focus only on the job posting.

Rules:
- Only state facts explicitly supported by the text. If salary, benefits, or location
  are not mentioned, write "Not stated" — do not guess or invent.
- Detect the primary language of the posting and write your summary in that language.
  If the posting is mixed ID/EN, you may use both briefly for key terms.
- Be concise and practical. A busy candidate should understand the role in under 60 seconds.
- If the text does not look like a job posting, say so clearly and explain what is missing.

Respond in markdown only. Do not wrap the markdown in a code block.

Use exactly this structure:

## Job at a glance
- **Role:**
- **Company:**
- **Location / work model:** (onsite, hybrid, remote — or Not stated)
- **Employment type:** (full-time, contract, internship, etc.)
- **Seniority level:** (intern, junior, mid, senior, lead — infer from requirements if not explicit)
- **Salary / compensation:** (Not stated if absent)

## What you'll do
3–5 bullet points summarizing responsibilities.

## Requirements
### Must-have
- Bullet list of hard requirements (years of experience, degree, core skills)

### Nice-to-have
- Bullet list, or "None stated"

## Skills & tools mentioned
Comma-separated list of technologies, tools, languages, or domains mentioned.

## Who this is a good fit for
1–2 sentences on ideal candidate profile.

## Possible gaps / red flags
- Missing info, unrealistic requirements, vague scope, or things a candidate should clarify
  (only if supported by the text; otherwise write "None obvious from posting")

## Apply checklist
- [ ] 3–5 concrete preparation steps before applying (tailor CV, portfolio, questions to ask, etc.)
"""

user_prompt_prefix = """
Analyze the following job posting content and produce the structured summary.

If company name or job title appear in the page title or headings, use them.
Prioritize requirements over marketing language.

Job posting content:
"""

In [9]:
def messages_for(job_posting_text: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + job_posting_text},
    ]


def summarize_job_posting(job_posting_text: str) -> str:
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(job_posting_text),
    )
    return response.choices[0].message.content


def display_job_summary(job_posting_text: str) -> None:
    display(Markdown(summarize_job_posting(job_posting_text)))

## Demo 1 — paste job posting text

Most reliable approach. Copy the job description from any platform and paste it below.
This example mimics a typical Indonesian tech listing (mixed ID/EN).

In [10]:
SAMPLE_JOB_POSTING = """
Backend Engineer (Payments) — Tokopedia
PT Tokopedia
Jakarta Selatan, DKI Jakarta · Hybrid · Full-time

Tentang peran ini:
Kami mencari Backend Engineer untuk bergabung dengan tim Payments. Kamu akan
membangun dan maintain layanan transaksi yang melayani jutaan pengguna setiap hari.

Tanggung jawab:
- Design, build, and operate scalable microservices in Go
- Collaborate with product and mobile teams on payment flows
- Improve reliability, monitoring, and incident response for critical paths
- Review code and mentor junior engineers

Kualifikasi:
- S1 Informatika atau bidang terkait
- Minimal 2 tahun pengalaman backend development
- Strong experience with Go or Java
- Familiar with PostgreSQL, Redis, and message queues (Kafka preferred)
- Good communication in English and Bahasa Indonesia

Nice to have:
- Experience in fintech or e-commerce payments
- Knowledge of PCI-DSS or security best practices
- Prior experience with Kubernetes

Benefit: BPJS, asuransi kesehatan, flexible hours, learning budget
Gaji: competitive, discuss at interview
"""

display_job_summary(SAMPLE_JOB_POSTING)

## Job at a glance
- **Role:** Backend Engineer (Payments)
- **Company:** PT Tokopedia
- **Location / work model:** Jakarta Selatan, DKI Jakarta / Hybrid
- **Employment type:** Full-time
- **Seniority level:** Mid (minimum 2 years experience)
- **Salary / compensation:** Competitive, discussed at interview

## What you'll do
- Design, build, and operate scalable microservices using Go
- Collaborate with product and mobile teams on payment flows
- Improve reliability, monitoring, and incident response for critical systems
- Review code and mentor junior engineers

## Requirements
### Must-have
- Bachelor's degree in Informatics or related field
- Minimum 2 years backend development experience
- Strong experience with Go or Java programming languages
- Familiarity with PostgreSQL, Redis, and message queues (Kafka preferred)
- Good communication skills in both English and Bahasa Indonesia

### Nice-to-have
- Experience in fintech or e-commerce payment systems
- Knowledge of PCI-DSS or security best practices
- Prior experience with Kubernetes

## Skills & tools mentioned
Go, Java, PostgreSQL, Redis, Kafka, Kubernetes, microservices architecture, PCI-DSS, English, Bahasa Indonesia

## Who this is a good fit for
A backend developer with solid experience in Go or Java who is interested in payment systems and scalable microservices, comfortable working in a hybrid environment, and willing to mentor juniors and collaborate across teams.

## Possible gaps / red flags
None obvious from posting

## Apply checklist
- [ ] Tailor CV to highlight backend development experience, especially in Go or Java
- [ ] Prepare examples or stories of building/operating scalable microservices
- [ ] Review basics of PostgreSQL, Redis, Kafka, and cloud/container orchestration (Kubernetes)
- [ ] Be ready to discuss experience or interest in fintech/payment systems
- [ ] Prepare to converse comfortably in English and Bahasa Indonesia

## Demo 2 — English posting (international)

Same tool, different language, the model detects and responds accordingly.

In [11]:
SAMPLE_JOB_POSTING_EN = """
Junior Data Analyst — Stripe
Remote (US time zones) · Full-time

About the role:
Join the Revenue Operations team to turn product and billing data into insights
for leadership. You will own recurring reports and ad-hoc analysis.

What you'll do:
- Write SQL queries against our warehouse (Snowflake)
- Build dashboards in Looker for finance and product partners
- Document metrics definitions and data quality issues

Requirements:
- 0–2 years experience in analytics, consulting, or a quantitative role
- Proficiency in SQL and spreadsheet modeling
- Clear written communication

Nice to have: Python, experiment design, Stripe or SaaS domain knowledge
Compensation: $85,000–$105,000 USD base + equity
"""

display_job_summary(SAMPLE_JOB_POSTING_EN)

## Job at a glance
- **Role:** Junior Data Analyst  
- **Company:** Stripe  
- **Location / work model:** Remote (US time zones)  
- **Employment type:** Full-time  
- **Seniority level:** Junior (0–2 years experience)  
- **Salary / compensation:** $85,000–$105,000 USD base + equity  

## What you'll do
- Write SQL queries against Snowflake data warehouse  
- Build Looker dashboards for finance and product teams  
- Document metric definitions and data quality issues  

## Requirements
### Must-have
- 0–2 years experience in analytics, consulting, or a quantitative role  
- Proficiency in SQL and spreadsheet modeling  
- Clear written communication skills  

### Nice-to-have
- Python  
- Experiment design knowledge  
- Experience in Stripe or SaaS domains  

## Skills & tools mentioned
SQL, Snowflake, Looker, spreadsheet modeling, Python, experiment design  

## Who this is a good fit for
An early-career analyst with foundational SQL and quantitative skills eager to grow in a data role supporting product and finance teams, ideally interested in SaaS or payments.

## Possible gaps / red flags
None obvious from posting  

## Apply checklist
- [ ] Tailor CV to highlight SQL, analytics, and quantitative experience  
- [ ] Prepare examples of past data projects or reports  
- [ ] Familiarize yourself with Stripe’s product and SaaS business model  
- [ ] Practice explaining data insights clearly in writing  
- [ ] Prepare questions about team structure and data tools usage

## Demo 3 — summarize from URL (optional)

Uses the same scraping approach as Day 1 (`requests` + BeautifulSoup).

**Limitations:** Many job boards render with JavaScript (LinkedIn, JobStreet, Glints) or block bots.
If scraping returns little text, paste the posting manually (Demo 1) or use a Selenium/Playwright scraper
from other community contributions.

Greenhouse and Lever public job pages often work with simple HTTP scraping.

In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    )
}


def fetch_job_posting_contents(url: str, max_chars: int = 4_000) -> str:
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else "No title found"
    if soup.body:
        for tag in soup.body(["script", "style", "img", "input", "nav", "footer"]):
            tag.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:max_chars]


def summarize_job_posting_url(url: str) -> str:
    return summarize_job_posting(fetch_job_posting_contents(url))


def display_job_summary_url(url: str) -> None:
    display(Markdown(summarize_job_posting_url(url)))

In [ ]:

url = "https://www.example.com"  

if url != "https://www.example.com":
    display_job_summary_url(url)
else:
    print("Set `url` to a job posting link, then re-run this cell.")

Set `url` to a job posting link, then re-run this cell.


## Your turn

Paste a job posting you are considering below, or pass a URL if scraping works for that site.

In [14]:
my_job_posting = """
Paste your job posting here...
"""

if "Paste your job posting here" not in my_job_posting:
    display_job_summary(my_job_posting)